# EditSVG Kaggle Evaluation Notebook (Friend's Edition)
Run this on Kaggle using a **T4x2 GPU** accelerator.

In [ ]:
!git clone https://github.com/ultraviolent07/EditSVG
!cd EditSVG && pip install -e .
!pip install vllm cairosvg>=2.7 Pillow>=9 torch torch-geometric sentence-transformers pydantic openai
!cat << 'EOF' > patch_flashinfer.py
import os, glob
try:
    import flashinfer
    fi_path = os.path.dirname(flashinfer.__file__)
    for fp in glob.glob(f'{fi_path}/**/*.py', recursive=True):
        try:
            with open(fp) as f: src = f.read()
            new_src = src.replace("'--compress-mode=size'", '""').replace('"--compress-mode=size"', '""')
            if new_src != src:
                with open(fp, 'w') as f: f.write(new_src)
                print(f"Patched {fp}")
        except: pass
except ImportError:
    print("flashinfer not installed, skipping patch")
EOF
!python patch_flashinfer.py

In [ ]:
import json

v5_prompt = """You are an SVG patch planner using protocol svgpatchlab.patch.v$version.
Return exactly one JSON object and nothing else. Do not output SVG, XML,
Markdown, analysis, or commentary.

Required output shape:
- one JSON object
- top-level integer field: version = 1
- top-level array field: operations
- each operation has:
  - op: "set_attributes"
  - targets: an array of actual node IDs copied from the actual context
  - attributes: an object whose keys are actual SVG attribute names and whose
    values are the computed replacement values

Strict JSON rules:
1. The top-level object may contain only `version` and `operations`.
2. Every edit must be an object inside the `operations` array.
3. Never put `op`, `targets`, `attributes`, or `names` at the top level.
4. Use `set_attributes` only for the edit tasks below.
5. Do not output any operation other than `set_attributes`.
6. Do not output SVG code or XML.
7. Do not output placeholder words. Every target must be a real node ID such as
   the IDs found in the actual context. Every attribute name must be a real SVG
   attribute such as fill, stroke, stroke-width, opacity, transform, or viewBox.

Planning checklist:
1. Read the actual instruction and context. Infer the requested edit type from
   the words in the instruction, not from any prior example or memory.
2. Use only node IDs that appear in the actual context. The SVG root is usually
   n0; do not target n0 unless the edit is truly a whole-image edit or the root
   itself directly has the attribute being changed.
3. Make the smallest patch that performs the requested edit.
4. Never modify path data (`d`) or polygon/polyline coordinate data (`points`).
5. Do not insert new shapes. Modify existing SVG elements only.
6. Visual context resolution: some nodes in the context contain a
   `visual_context` field with a `summary` (short phrase), `labels` (list of
   semantic terms like "window", "eye", "roof", "wheel", "background"), and a `role`.
   - Use the visual context strictly to identify semantic objects. Do not infer or invent SVG attributes from the summary text.
   - If the instruction refers to a part by name (e.g. "windows", "roof",
     "eyes", "door"), first check each node's `visual_context.labels` and
     `visual_context.summary` for a match. Target those nodes.
   - If the instruction asks to edit a specific letter, word, or punctuation
     mark, explicitly check the `visual_context.summary` for exact OCR text 
     matches (e.g. 'UP!') and target those nodes.
   - If the instruction generically refers to the "base", "main body", "structure",
     or an object without specifying a sub-part, prioritize targeting nodes 
     that have `"role": "primary_fill"`. Do NOT target `"role": "part"`, 
     `"highlight"`, or `"shadow"` for these root object edits.
   - If no `visual_context` is present on any node, fall back to matching by
     `fill` color or structural position as usual.
   - `visual_context` labels take priority over color matching when both are
     available and the instruction names a part rather than a color.

Generic SVG edit rules:
- Color/recolor/fill changes:
  * If the instruction says to change the part with OLD_COLOR to NEW_COLOR,
    find nodes whose direct `fill` or resolved `fill` equals OLD_COLOR.
  * Set only `fill` to NEW_COLOR on those matching nodes.
  * Do not set `fill` to OLD_COLOR. You must only modify the color; do not modify structure or opacity.
  * Do not target n0 for a color change unless n0 itself directly has the
    OLD_COLOR fill.
- Outline/contour/border changes:
  * Target the nodes indicated by the instruction, usually nodes whose direct
    or resolved `fill` equals the named source color.
  * Set `stroke` and `stroke-width`.
  * If the outline color is not specified, use black.
  * Preserve existing `fill`; do not recolor the filled interior.
- Transparency/opacity changes:
  * Target exactly n0 for an entire-image transparency edit.
  * You must only output the `opacity` attribute. 
  * Any structural edits or color edits are strictly forbidden here. You are strictly limited to outputting exactly one attribute: opacity. If you output any other attribute name, the system will crash. do not output viewbox for transparency changes
- Crop/trim/keep-half changes:
  * Target exactly n0 and modify only the root `viewBox`.
  * Use the actual root viewBox from the context. If it is `x y w h`, then:
    - keep left half: `x y w/2 h`
    - keep right half: `x+w/2 y w/2 h`
    - keep top half: `x y w h/2`
    - keep bottom half: `x y+h/2 w h/2`
  * Output computed numeric values, not formulas like `w/2` or `x+w/2`.
  * Preserve the unchanged viewBox components exactly.
- Upside-down / vertical flip changes:
  * Target exactly n0 and set only `transform`.
  * If the root viewBox is `x y w h` and y is 0, use:
    `translate(0,h) scale(1,-1)`, replacing h with the actual numeric height.
  * Any color or viewBox edits are strictly forbidden here.

Compute every target, color, number, and attribute value from the actual
instruction and actual context below.

Actual edit instruction:
$instruction

Actual $context_name:
$context

Output the JSON patch object now.
"""

# 1. Inject the V5 prompt directly over the existing patch_v3 template to hotfix it
with open("EditSVG/svgpatchlab/prompt_templates/patch_v3.txt", "w") as f:
    f.write(v5_prompt)

# 2. Fix max_tokens bug just in case he hasn't yet
try:
    with open("EditSVG/configs/models/qwen3.5-4b-openai.json", "r") as f:
        model_config = json.load(f)
    if model_config.get("max_tokens") == 512:
        model_config["max_tokens"] = 4096
        with open("EditSVG/configs/models/qwen3.5-4b-openai.json", "w") as f:
            json.dump(model_config, f, indent=2)
except FileNotFoundError:
    pass

print("Patched prompt and configs successfully!")

In [ ]:
import subprocess
import time
import urllib.request
import urllib.error
import sys

print("--> Starting vLLM API server in the background...")
vllm_cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", "Qwen/Qwen3.5-4B",
    "--port", "8000",
    "--enforce-eager",
    "--max-model-len", "8192" # Limit max length to fit inside Kaggle T4 RAM
]
server = subprocess.Popen(vllm_cmd, stdout=sys.stdout, stderr=sys.stderr)

print("--> Waiting for server to boot (this takes ~2-3 minutes)...")
max_retries = 60
for i in range(max_retries):
    try:
        req = urllib.request.Request("http://localhost:8000/v1/models")
        with urllib.request.urlopen(req) as response:
            if response.status == 200:
                print("--> Server is UP!")
                break
    except urllib.error.URLError:
        pass
    
    if server.poll() is not None:
        raise RuntimeError("vLLM server crashed!")
        
    time.sleep(10)
else:
    server.terminate()
    raise TimeoutError("vLLM server took too long to start.")

print("--> Starting standard baseline evaluation...")
eval_cmd = [
    sys.executable, "-m", "svgpatchlab.cli", "evaluate",
    "--config", "configs/experiments/skeleton_patch.json"
]
subprocess.run(eval_cmd, cwd="EditSVG", check=True)

server.terminate()
print("--> Finished! Check EditSVG/runs/ for the output!")